# 13.1 - AI App Costs

**Module 13 - Cost & Performance** | Netsetos GenAI Engineering

This notebook builds the cost model and tooling to know every token before you spend it: a per-token calculator, a single-request pricer, a hidden-multiplier estimator, a discount-stacker, a unit-economics projector, a per-call cost meter, and a spend dashboard with an optimizer. Every block is a plain-Python model with illustrative prices - no API key required - so you can see exactly where an LLM bill comes from.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

This notebook is deliberately keyless: every cost model runs on plain arithmetic with illustrative prices, so there is nothing to install to work through the concepts. The only external dependency, `anthropic`, is needed just for the illustrative per-call meter shown later, so the install line is left commented out.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

Environment prep, not a computation. The install line is commented out because none of the runnable blocks call an API; the remaining line is a note that the lesson uses fixed, seeded values throughout so your output will match the printed comments.

**How the code works - step by step**
- **Commented `pip install anthropic`** - uncomment only if you later want to run the illustrative metering wrapper against the real SDK.
- **The reproducibility comment** - a reminder that every block uses fixed token counts and illustrative prices, so runs are deterministic.

**In one line:** nothing to install - the cost models are pure Python.

**What you'll see:** (no output - environment prep)

## 1 - Tokens, not requests

The single fact that resets your intuition: **cost is tokens times price, not calls times price**. Traditional SaaS bills a flat rate per request; an LLM bills per token, with input and output priced separately and output costing more. This block puts a flat SaaS meter next to a per-token meter so you can watch them diverge.

In [ ]:
# ILLUSTRATIVE prices (USD per 1M tokens; check provider docs for live numbers).
# Two facts: input and output are priced separately, and output costs more.
PRICES = {  # tier: (input_per_1M, output_per_1M)
    "flash":    (0.15,  0.60),   # small / fast
    "frontier": (3.00, 15.00),   # top-tier (output is 5x input)
}
def cost(tier, in_tok, out_tok):
    pin, pout = PRICES[tier]
    return (in_tok * pin + out_tok * pout) / 1_000_000

# Traditional SaaS: a flat rate per REQUEST.
print("Traditional SaaS: $0.01 per call -> 10,000 calls = ${:.0f}. Simple and predictable.".format(10000 * 0.01))
print()
print("LLM: cost = tokens x price. A 100-token query vs a 10,000-token document, same model (flash):")
a = cost("flash", 100, 200)
b = cost("flash", 10000, 200)
print("  query    (in 100,   out 200): ${:.6f}".format(a))
print("  document (in 10000, out 200): ${:.6f}   <- its INPUT alone costs 100x the query's input".format(b))
print()
print("Same request (in 100, out 200) across model tiers:")
for tier in ["flash", "frontier"]:
    print("  {:<9} ${:.6f}".format(tier, cost(tier, 100, 200)))
print("  frontier is {:.0f}x the flash cost - and the full market spans a ~600x range.".format(cost("frontier",100,200)/cost("flash",100,200)))
print()
print("Cost is tokens x price, NOT calls x price. A long answer or a big context costs far more.")

# Output:
# Traditional SaaS: $0.01 per call -> 10,000 calls = $100. Simple and predictable.
#
# LLM: cost = tokens x price. A 100-token query vs a 10,000-token document, same model (flash):
#   query    (in 100,   out 200): $0.000135
#   document (in 10000, out 200): $0.001620   <- its INPUT alone costs 100x the query's input
#
# Same request (in 100, out 200) across model tiers:
#   flash     $0.000135
#   frontier  $0.003300
#   frontier is 24x the flash cost - and the full market spans a ~600x range.
#
# Cost is tokens x price, NOT calls x price. A long answer or a big context costs far more.

**Explanation**

A cost calculator that contrasts two billing models. It defines a price table (input and output rate per million tokens for a small tier and a frontier tier) and a `cost()` function, then prints the same workload three ways: flat SaaS, the same model on a tiny vs a huge prompt, and the same request across model tiers. The point is comparison, not a model call.

**How the code works - step by step**
- **`PRICES` dict** - maps each tier to `(input_per_1M, output_per_1M)`; note frontier output is 5x its input, and 25x flash's output.
- **`cost(tier, in_tok, out_tok)`** - the core formula: `(in_tok x input_price + out_tok x output_price) / 1_000_000`.
- **SaaS line** - 10,000 calls at a flat $0.01 = a predictable $100, independent of work done.
- **Query vs document** - same flash model, 100 vs 10,000 input tokens: the document's input alone costs 100x the query's.
- **Tier loop** - the same (100 in, 200 out) request on flash vs frontier, then the ratio between them.

**In one line:** tokens x price, so a big context or a pricier tier costs far more for the 'same' call.

**What you'll see:** Prints the $100 SaaS baseline, then flash query $0.000135 vs document $0.001620 (100x the input), then flash $0.000135 vs frontier $0.003300 - frontier is 24x flash, inside a market that spans roughly 600x.

## 2 - The cost of one request

Price a single request exactly: `input_tokens x input_price + output_tokens x output_price`. Two surprises - output can cost more than input even with fewer tokens (it is priced higher), and the input you did not write (a RAG call's retrieved context) is usually the bulk of what you pay. This block prices a plain chat request and a RAG request side by side.

In [ ]:
# The cost of ONE request = input cost + output cost. Output is priced higher, so it can dominate.
PRICES = {"flash": (0.15, 0.60), "frontier": (3.00, 15.00)}
def parts(tier, in_tok, out_tok):
    pin, pout = PRICES[tier]
    ic = in_tok * pin / 1_000_000
    oc = out_tok * pout / 1_000_000
    return ic, oc, ic + oc

ic, oc, total = parts("flash", 500, 300)
print("A chat request on flash (in 500, out 300):")
print("  input cost  ${:.6f}   (500 tokens x $0.15/M)".format(ic))
print("  output cost ${:.6f}   (300 tokens x $0.60/M)".format(oc))
print("  total       ${:.6f}   <- output cost > input cost despite FEWER output tokens (output is 4x)".format(total))
print()
# RAG: the input you did NOT write. A tiny query rides with a big retrieved context.
query, context, out = 50, 4000, 400
ic2, oc2, total2 = parts("flash", query + context, out)
print("A RAG request on flash (query {}, retrieved context {}, out {}):".format(query, context, out))
print("  total input tokens: {} (query {} + context {})".format(query + context, query, context))
print("  context is {:.0f}% of the input - the 'hidden' cost you pay on EVERY call".format(context / (query + context) * 100))
print("  total cost ${:.6f}   (you thought you sent {} tokens; you sent {})".format(total2, query, query + context))
print()
print("Count the WHOLE prompt: system prompt + retrieved context + history + query, not just what the user typed.")

# Output:
# A chat request on flash (in 500, out 300):
#   input cost  $0.000075   (500 tokens x $0.15/M)
#   output cost $0.000180   (300 tokens x $0.60/M)
#   total       $0.000255   <- output cost > input cost despite FEWER output tokens (output is 4x)
#
# A RAG request on flash (query 50, retrieved context 4000, out 400):
#   total input tokens: 4050 (query 50 + context 4000)
#   context is 99% of the input - the 'hidden' cost you pay on EVERY call
#   total cost $0.000847   (you thought you sent 50 tokens; you sent 4050)
#
# Count the WHOLE prompt: system prompt + retrieved context + history + query, not just what the user typed.

**Explanation**

A per-request pricer that splits every cost into its input and output halves. The `parts()` helper returns `(input_cost, output_cost, total)` so the block can show that a concise answer is not automatically cheap, and that a 50-token question dragging 4,000 tokens of context makes the context ~99% of the billable input.

**How the code works - step by step**
- **`parts(tier, in_tok, out_tok)`** - returns input cost, output cost, and their sum separately so you can see which half dominates.
- **Chat request** - flash with 500 in / 300 out: output cost ($0.000180) exceeds input cost ($0.000075) despite fewer output tokens, because output is 4x the price.
- **RAG request** - a 50-token query plus 4,000 tokens of retrieved context; computes context as a percentage of total input.
- **The reveal** - you thought you sent 50 tokens; you sent 4,050, and pay for all of them on every call.

**In one line:** count the whole prompt (system + context + history + query), and remember output is priced higher.

**What you'll see:** Chat: input $0.000075, output $0.000180, total $0.000255 (output wins). RAG: 4,050 input tokens, context = 99% of input, total $0.000847 - the hidden cost you pay on every call.

## 3 - The hidden multipliers

Even after counting the whole prompt, the real bill is often several times a naive estimate - because of tokens that are billed but invisible. A reasoning model bills a long internal chain-of-thought as output you never see, and an agent fans one user request into many LLM calls. This block quantifies both multipliers.

In [ ]:
# The bill is often several times a naive estimate, because of tokens you never see.
PRICES = {"frontier": (3.00, 15.00)}
def cost(in_tok, out_tok):
    pin, pout = PRICES["frontier"]
    return (in_tok * pin + out_tok * pout) / 1_000_000

# (1) REASONING tokens: billed as output, but hidden from you.
visible_out, reasoning, in_tok = 500, 3000, 1000
naive = cost(in_tok, visible_out)                 # what you'd estimate from the visible answer
real = cost(in_tok, visible_out + reasoning)      # what you're actually billed
print("Reasoning model - a one-line answer:")
print("  visible output tokens:  {}".format(visible_out))
print("  hidden reasoning tokens: {}  (billed as output, never shown)".format(reasoning))
print("  billed output = {} -> {}x the visible output".format(visible_out + reasoning, (visible_out + reasoning) // visible_out))
print("  naive estimate ${:.6f}   real cost ${:.6f}   = {:.1f}x more than you'd guess".format(naive, real, real / naive))
print()
# (2) AGENT loops: one user request fans out into many LLM calls.
calls = [("plan", 800, 200), ("tool call", 900, 150), ("tool call", 950, 150), ("reflect", 1100, 200), ("answer", 1200, 400)]
print("Agent - one user request fans out into {} LLM calls:".format(len(calls)))
run_total = 0.0
for name, i, o in calls:
    c = cost(i, o); run_total += c
    print("  {:<10} in {:>4} out {:>3}  ${:.6f}".format(name, i, o, c))
single = cost(800, 200)
print("  one user request costs ${:.6f} = {:.1f}x a single call - the agent multiplier".format(run_total, run_total / single))
print()
print("Estimate from the ACTUAL token usage (reasoning + all calls), never from the visible text.")

# Output:
# Reasoning model - a one-line answer:
#   visible output tokens:  500
#   hidden reasoning tokens: 3000  (billed as output, never shown)
#   billed output = 3500 -> 7x the visible output
#   naive estimate $0.010500   real cost $0.055500   = 5.3x more than you'd guess
#
# Agent - one user request fans out into 5 LLM calls:
#   plan       in  800 out 200  $0.005400
#   tool call  in  900 out 150  $0.004950
#   tool call  in  950 out 150  $0.005100
#   reflect    in 1100 out 200  $0.006300
#   answer     in 1200 out 400  $0.009600
#   one user request costs $0.031350 = 5.8x a single call - the agent multiplier
#
# Estimate from the ACTUAL token usage (reasoning + all calls), never from the visible text.

**Explanation**

A two-part estimator that contrasts a naive guess (from the visible answer) with the real bill (from actual usage). Part 1 adds hidden reasoning tokens onto the visible output; part 2 sums the cost of an agent's five-call fan-out and compares it to a single call. Both use the frontier price table.

**How the code works - step by step**
- **`cost(in_tok, out_tok)`** - frontier-only pricer reused throughout.
- **Reasoning block** - `naive` prices just the 500 visible output tokens; `real` prices 500 + 3,000 hidden reasoning tokens, then reports both the output ratio (7x) and the cost ratio.
- **`calls` list** - five labelled steps (plan, two tool calls, reflect, answer) with their own token counts.
- **Agent loop** - sums each step's cost into `run_total`, then divides by a single `plan`-sized call to get the agent multiplier.

**In one line:** estimate from the API's reported usage (reasoning + every call), never from the visible text.

**What you'll see:** Reasoning: billed output is 3,500 (7x visible), so real cost $0.055500 vs naive $0.010500 = 5.3x more. Agent: five calls total $0.031350 = 5.8x a single call.

## 4 - The three big discounts

The bill is not fixed - three levers cut it hard and two of them stack: prompt caching (a reusable prefix at ~10% of input price), the batch API (~50% off async work), and model tiering (route easy queries to a small model). This block stacks the discounts on one request and prices the tiering lever separately.

In [ ]:
# Three levers cut the bill hard, and they STACK. (ILLUSTRATIVE rates.)
IN, OUT = 3.00, 15.00           # frontier $/1M
CACHED_IN = IN * 0.10           # cached input ~10% of input price (up to ~90% off)
BATCH = 0.50                    # batch API ~50% off everything

# A request with a big reusable prefix (a stable system prompt / context) + a little fresh input.
cacheable, fresh_in, out = 3500, 500, 400
def total(cached, batch):
    if cached:
        in_cost = (cacheable * CACHED_IN + fresh_in * IN) / 1_000_000
    else:
        in_cost = (cacheable + fresh_in) * IN / 1_000_000
    c = in_cost + out * OUT / 1_000_000
    return c * (BATCH if batch else 1.0)

std = total(False, False)
print("A request (reusable prefix {} + fresh input {} + output {}) on frontier:".format(cacheable, fresh_in, out))
print("  standard                 ${:.6f}".format(std))
print("  + prompt caching         ${:.6f}   ({:.0f}% of standard)".format(total(True, False),  total(True, False)  / std * 100))
print("  + batch (async)          ${:.6f}   ({:.0f}% of standard)".format(total(False, True),  total(False, True)  / std * 100))
print("  + caching AND batch      ${:.6f}   ({:.0f}% of standard)  <- the discounts stack".format(total(True, True), total(True, True) / std * 100))
print()
# The third lever: model tiering (route the easy queries down).
flash = (500 * 0.15 + 400 * 0.60) / 1_000_000
front = (500 * 3.00 + 400 * 15.00) / 1_000_000
print("Model tiering - the same short query on flash vs frontier:")
print("  flash ${:.6f}  vs  frontier ${:.6f}  = {:.0f}x cheaper (if the small model is good enough)".format(flash, front, front / flash))
print()
print("Cache the reusable prefix, batch the async work, tier by difficulty - stacked, ~a quarter of the standard bill.")

# Output:
# A request (reusable prefix 3500 + fresh input 500 + output 400) on frontier:
#   standard                 $0.018000
#   + prompt caching         $0.008550   (48% of standard)
#   + batch (async)          $0.009000   (50% of standard)
#   + caching AND batch      $0.004275   (24% of standard)  <- the discounts stack
#
# Model tiering - the same short query on flash vs frontier:
#   flash $0.000315  vs  frontier $0.007500  = 24x cheaper (if the small model is good enough)
#
# Cache the reusable prefix, batch the async work, tier by difficulty - stacked, ~a quarter of the standard bill.

**Explanation**

A discount-stacker built around one request with a big reusable prefix. The `total(cached, batch)` function recomputes the same request under each combination of the two stacking discounts, so you can read each lever's effect as a percentage of the standard bill; a final snippet compares the same short query on flash vs frontier for the tiering lever.

**How the code works - step by step**
- **Rate constants** - `CACHED_IN` is 10% of the input rate; `BATCH` is a 0.50 multiplier on the whole cost.
- **`total(cached, batch)`** - prices the reusable prefix at the cached rate when caching is on, then halves the result when batch is on.
- **Four prints** - standard, +caching, +batch, +both, each with its percentage of standard, showing the discounts compound.
- **Tiering snippet** - the same short query on flash vs frontier, and the ratio between them.

**In one line:** cache the reusable prefix, batch the async work, tier by difficulty - stacked, ~a quarter of the standard bill.

**What you'll see:** Standard $0.018000; caching alone 48%, batch alone 50%, both together $0.004275 = 24% of standard. Tiering: the same query is $0.000315 on flash vs $0.007500 on frontier - 24x cheaper.

## 5 - Cost at scale and unit economics

A per-request cost is abstract; multiply it out and it gets real. This block projects one request to a daily and monthly bill, then to the number that decides the business - cost per user - and exposes two hard truths: spend is heavily skewed toward a few users, and AI products run ~50-60% gross margin versus 80-90% for classic SaaS.

In [ ]:
# Project one request to a monthly bill, then to the number that decides the business: cost per user.
PRICES = {"balanced": (1.00, 5.00)}
def cost(in_tok, out_tok):
    pin, pout = PRICES["balanced"]
    return (in_tok * pin + out_tok * pout) / 1_000_000

per_req = cost(1500, 500)   # a RAG-ish request
daily_requests = 20000
monthly = per_req * daily_requests * 30
print("Project to scale (balanced model, in 1500 / out 500):")
print("  per request  ${:.6f}".format(per_req))
print("  per day      ${:>8.2f}   ({} requests)".format(per_req * daily_requests, daily_requests))
print("  per month    ${:>8.2f}".format(monthly))
print()
# Cost per user - and the number is SKEWED. A few users drive most of the spend.
users = 600                        # a heavy-usage AI product where inference is real COGS
calls_share = [0.60, 0.25, 0.15]   # top 5% of users / next 15% / everyone else
print("Cost per user is an average that HIDES the skew ({} active users):".format(users))
print("  average cost per user     ${:.4f}/month".format(monthly / users))
top5pct_users = int(users * 0.05)
print("  but the top 5% ({} users) drive {:.0f}% of the spend = ${:.2f}/month among them".format(top5pct_users, calls_share[0] * 100, monthly * calls_share[0]))
print("  -> watch the outliers (p99), not just the mean")
print()
# Gross margin: is the feature even profitable?
revenue_per_user, cost_per_user = 10.00, monthly / users
margin = (revenue_per_user - cost_per_user) / revenue_per_user * 100
print("Unit economics per user: revenue ${:.2f} - cost ${:.4f} -> gross margin {:.0f}%".format(revenue_per_user, cost_per_user, margin))
print("AI products run ~50-60% gross margin (vs 80-90% for classic SaaS) - inference is real COGS.")

# Output:
# Project to scale (balanced model, in 1500 / out 500):
#   per request  $0.004000
#   per day      $   80.00   (20000 requests)
#   per month    $ 2400.00
#
# Cost per user is an average that HIDES the skew (600 active users):
#   average cost per user     $4.0000/month
#   but the top 5% (30 users) drive 60% of the spend = $1440.00/month among them
#   -> watch the outliers (p99), not just the mean
#
# Unit economics per user: revenue $10.00 - cost $4.0000 -> gross margin 60%
# AI products run ~50-60% gross margin (vs 80-90% for classic SaaS) - inference is real COGS.

**Explanation**

A unit-economics projector. It prices one RAG-ish request on a balanced tier, scales it by daily volume and 30 days, divides by active users for an average, then shows how the top 5% of users own most of that spend, and finally computes gross margin from revenue minus cost per user.

**How the code works - step by step**
- **`per_req`** - one 1,500-in / 500-out request on the balanced model.
- **Projection** - multiplies by 20,000 daily requests, then by 30 for the monthly bill.
- **Average per user** - monthly bill / 600 users.
- **Skew** - `calls_share[0]` says the top 5% (30 users) drive 60% of spend; prints their share of the monthly total.
- **Gross margin** - `(revenue_per_user - cost_per_user) / revenue_per_user`, with revenue at $10.

**In one line:** know cost per user and margin before you scale, and watch the p99 tail, not the mean.

**What you'll see:** Per request $0.004000, per day $80.00, per month $2400.00. Average $4.00/user, but the top 30 users drive $1440.00 of it; revenue $10 - cost $4 = 60% gross margin.

## 6 - Cost attribution: meter every call

A provider dashboard gives you one number - the total bill - and cannot tell you which team, feature, or customer drove it. You cannot optimize what you cannot attribute, so this block tags every call with metadata and rolls spend up by any dimension, then tracks per-agent-run cost at the median and the p99.

In [ ]:
# You cannot optimize what you cannot attribute. Tag every call, then roll up by any dimension.
calls = [  # (team, feature, model, cost_usd, agent_run_id)
    ("product",  "chat",    "flash",    0.0003, "run-1"),
    ("product",  "chat",    "flash",    0.0004, "run-1"),
    ("product",  "summary", "frontier", 0.0180, "run-2"),
    ("research", "agent",   "frontier", 0.0090, "run-3"),
    ("research", "agent",   "frontier", 0.0110, "run-3"),
    ("research", "agent",   "frontier", 0.0120, "run-3"),
    ("research", "agent",   "frontier", 0.2400, "run-4"),   # a runaway loop
    ("product",  "chat",    "flash",    0.0003, "run-5"),
]
def rollup(key_i):
    d = {}
    for c in calls:
        d[c[key_i]] = d.get(c[key_i], 0.0) + c[3]
    return d

print("Spend rolled up by TEAM:")
for k, v in sorted(rollup(0).items(), key=lambda kv: -kv[1]):
    print("  {:<10} ${:.4f}".format(k, v))
print("Spend rolled up by FEATURE:")
for k, v in sorted(rollup(1).items(), key=lambda kv: -kv[1]):
    print("  {:<10} ${:.4f}".format(k, v))
print()
# Per agent-run cost: median shows normal, p99 exposes runaway loops.
runs = {}
for c in calls:
    runs[c[4]] = runs.get(c[4], 0.0) + c[3]
run_costs = sorted(runs.values())
def pctl(vals, p):
    k = max(0, int(round((p / 100.0) * (len(vals) - 1))))
    return sorted(vals)[k]
print("Per agent-run cost across {} runs:".format(len(run_costs)))
print("  median ${:.4f}   p99 ${:.4f}   <- p99 is {:.0f}x the median: a runaway run".format(pctl(run_costs, 50), pctl(run_costs, 99), pctl(run_costs, 99) / pctl(run_costs, 50)))
print()
print("Every call carries team / feature / model / tokens / cost / agent_run_id, so cost meets the business.")

# Output:
# Spend rolled up by TEAM:
#   research   $0.2720
#   product    $0.0190
# Spend rolled up by FEATURE:
#   agent      $0.2720
#   summary    $0.0180
#   chat       $0.0010
#
# Per agent-run cost across 5 runs:
#   median $0.0180   p99 $0.2400   <- p99 is 13x the median: a runaway run
#
# Every call carries team / feature / model / tokens / cost / agent_run_id, so cost meets the business.

**Explanation**

An attribution and roll-up model. It represents each call as a tagged tuple (team, feature, model, cost, agent_run_id), sums cost by any chosen field with a generic `rollup()`, then groups cost by run and computes median vs p99 to expose the one runaway loop a mean would smother.

**How the code works - step by step**
- **`calls` list** - eight tagged calls, one of them a deliberate runaway ($0.2400).
- **`rollup(key_i)`** - generic grouper: sums `cost` (index 3) keyed by whichever tuple field you pass, reused for team and feature views.
- **Run grouping** - sums cost per `agent_run_id` into `runs`.
- **`pctl(vals, p)`** - a nearest-rank percentile helper used for both the median (p50) and p99.
- **Final print** - median vs p99 per run and the ratio between them.

**In one line:** tag every call so cost meets the business, and watch the p99 to catch runaway runs.

**What you'll see:** Roll-ups: research $0.2720 vs product $0.0190 by team; agent $0.2720 dominates by feature. Per run: median $0.0180, p99 $0.2400 - the p99 is 13x the median, a runaway run.

## 7 - The cost dashboard and optimizer

The metered logs become a dashboard when you aggregate them into the questions that save money - spend by model, cache hit rate, expensive outliers - then an optimizer names the single biggest cost driver and recommends a concrete fix with an estimated saving. Teams that apply this discipline routinely cut LLM spend by half to two-thirds.

In [ ]:
# Aggregate the metered logs into the questions that save money, then name the biggest driver.
logs = [  # (team, model, category, cost_usd, tokens, cached)
    ("product",  "flash",    "fast",     0.0003, 800,  True),
    ("product",  "flash",    "fast",     0.0004, 900,  False),
    ("research", "frontier", "frontier", 0.0180, 900,  False),
    ("research", "frontier", "frontier", 0.0900, 950,  False),   # frontier on a SHORT query
    ("research", "frontier", "frontier", 0.0850, 900,  False),
    ("product",  "flash",    "fast",     0.0003, 200,  True),
]
total = sum(l[3] for l in logs)
by_model = {}
for l in logs:
    by_model[l[1]] = by_model.get(l[1], 0.0) + l[3]
cache_hits = sum(1 for l in logs if l[5])

print("Cost dashboard ({} calls, ${:.4f} total):".format(len(logs), total))
print("  spend by model:")
for m, v in sorted(by_model.items(), key=lambda kv: -kv[1]):
    print("    {:<9} ${:.4f}  ({:.0f}% of spend)".format(m, v, v / total * 100))
print("  cache hit rate: {:.0f}%".format(cache_hits / len(logs) * 100))
outliers = [l for l in logs if l[3] > 0.05]
print("  expensive outliers (> $0.05): {} calls - all frontier on short queries".format(len(outliers)))
print()
# The optimizer: the biggest driver + a concrete fix with an estimated saving.
front_spend = by_model.get("frontier", 0.0)
short_frontier = [l for l in logs if l[1] == "frontier" and l[4] < 1000]
saving = sum(l[3] for l in short_frontier) * 0.85    # ~85% cheaper on flash
print("Biggest cost driver: frontier is {:.0f}% of spend, mostly on SHORT queries.".format(front_spend / total * 100))
print("  RECOMMEND: route queries under 1K tokens to flash  ->  est. saving ${:.4f} (~85% of that spend)".format(saving))
print("Discipline, not new architecture: teams routinely cut LLM spend by half to two-thirds this way.")

# Output:
# Cost dashboard (6 calls, $0.1940 total):
#   spend by model:
#     frontier  $0.1930  (99% of spend)
#     flash     $0.0010  (1% of spend)
#   cache hit rate: 33%
#   expensive outliers (> $0.05): 2 calls - all frontier on short queries
#
# Biggest cost driver: frontier is 99% of spend, mostly on SHORT queries.
#   RECOMMEND: route queries under 1K tokens to flash  ->  est. saving $0.1641 (~85% of that spend)
# Discipline, not new architecture: teams routinely cut LLM spend by half to two-thirds this way.

**Explanation**

A dashboard-plus-optimizer over metered logs. It totals spend, groups it by model, counts cache hits, flags calls above a cost threshold, then diagnoses the biggest driver (frontier on short queries) and estimates the saving from routing those queries to the cheap tier - implementation discipline, not new architecture.

**How the code works - step by step**
- **`logs` list** - six tagged calls including cached flags and token counts; three are frontier on short queries.
- **`by_model`** - sums cost per model for the spend breakdown.
- **`cache_hits`** - counts logs where the cached flag is true, printed as a hit rate.
- **`outliers`** - filters calls above the $0.05 threshold.
- **Optimizer** - identifies frontier's share of spend, isolates sub-1K-token frontier calls, and estimates an ~85% saving from moving them to flash.

**In one line:** aggregate the meter, find the biggest driver, recommend the concrete fix.

**What you'll see:** Dashboard: $0.1940 total, frontier is 99% of spend vs flash 1%, cache hit rate 33%, 2 outliers above $0.05. Optimizer: route sub-1K-token queries to flash for an estimated $0.1641 saving.

Across seven models you traced the whole cost chain: an LLM bills per token not per request (1), one request is priced by input + output with RAG context dominating the input (2), hidden reasoning tokens and agent loops multiply the real bill (3), caching + batch + tiering cut it (4), and projecting to cost-per-user reveals margin and skew (5). The last two blocks turn cost into an operations discipline - meter every call so spend meets the team/feature/customer (6), then aggregate the meter into a dashboard that names your single biggest driver and its fix (7). From here the rest of Module 13 goes deeper on driving the bill down (model routing, quantization, self-hosting economics); the caching mechanics live in Lesson 12.4 and the trace your attribution rides on is in Lesson 12.8.